In [1]:
# Spark Session

from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Reading Complex Data Formats")
    .master("local[*]")
    .getOrCreate()
)

spark

In [2]:
# Read Parquet Sales data

df_parquet = spark.read.format("parquet").load("data/input/sales_data.parquet")
df_parquet = spark.read.format("parquet").load("data/input/sales_total_parquet/*.parquet")

In [3]:
df_parquet.printSchema()
df_parquet.show()

root
 |-- transacted_at: timestamp (nullable = true)
 |-- trx_id: integer (nullable = true)
 |-- retailer_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: integer (nullable = true)

+-------------------+----------+-----------+--------------------+-------+----------+
|      transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+-------------------+----------+-----------+--------------------+-------+----------+
|2017-12-27 19:00:00| 330765426|  887300947|Kroger   ccd id: ...|  33.56|2068475652|
|2017-11-26 21:00:00|1377679664| 1070485878|Amazon.com    ccd...|  24.43|1640819540|
|2017-12-12 23:00:00| 472018705| 2001148981|  unkn      Columbia|   1.24| 481821583|
|2017-05-19 19:00:00|1127671830|  847200066|            Wal-Mart|2155.48|2074005445|
|2017-11-17 21:00:00| 233137169|  847200066|            Wal-Mart|   4.13|2043825401|
|2017-12-15 12:00:00| 603124844|  887300947|Kroger   ccd id: .

In [4]:
# Read ORC Sales data

#df_orc = spark.read.format("orc").load("data/input/sales_total_orc/*.orc")
df_orc = spark.read.format("orc").load("data/input/sales_data.orc")

In [5]:
df_orc.printSchema()
df_orc.show(50)

root
 |-- transacted_at: timestamp (nullable = true)
 |-- trx_id: integer (nullable = true)
 |-- retailer_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: integer (nullable = true)

+-------------------+----------+-----------+--------------------+-------+----------+
|      transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+-------------------+----------+-----------+--------------------+-------+----------+
|2017-11-24 19:00:00|1995601912| 2077350195|Walgreen       11-25| 197.23| 216510442|
|2017-11-24 19:00:00|1734117021|  644879053|unkn    ppd id: 7...|   8.58| 930259917|
|2017-11-24 19:00:00|1734117022|  847200066|Wal-Mart  ppd id:...|1737.26|1646415505|
|2017-11-24 19:00:00|1734117030| 1953761884|Home Depot     pp...|  384.5| 287177635|
|2017-11-24 19:00:00|1734117089| 1898522855| Target        11-25|  66.33|1855530529|
|2017-11-24 19:00:00|1734117117|  997626433|Sears  ppd id: 85.

In [6]:
# Python decorator

import time

def get_time(func):
    def inner_get_time() -> str:
        start_time = time.time()
        func()
        end_time = time.time()
        return (f"Execution time: {(end_time - start_time)*1000} ms")
    return inner_get_time

In [7]:
# Columnar storage refers to storing data column-by-column instead of row-by-row

@get_time
def x():
    df = spark.read.format("parquet").load("data/input/sales_data.parquet")
    print(df.count())

x()

1102576


'Execution time: 1225.2092361450195 ms'

In [8]:
@get_time
def x():
    df = spark.read.format("parquet").load("data/input/sales_data.parquet")
    print(df.select("trx_id").count())

x()

# Column storage is more efficient because it reads only needed columns

#df = spark.read.format("parquet").load("data/input/sales_data.parquet")
#df.select("trx_id").count()

1102576


'Execution time: 671.4894771575928 ms'

In [9]:
# Row storage in Apache Spark refers to how data is physically laid out when stored or processed in a row-oriented format

@get_time 
def x():
    df = spark.read.format("csv").load("data/input/emp_new.csv")
    print(df.count())
    
x()

21


'Execution time: 1552.3502826690674 ms'

In [10]:
# RECURSIVE READ

# sales_recursive
# |__ sales_1\1.parquet
# |__ sales_1\sales_2\2.parquet

In [11]:
df_1 = spark.read.format("parquet").load("data/input/sales_recursive/sales_1/1.parquet")
df_1.show()

+-------------------+----------+-----------+--------------------+------+---------+
|      transacted_at|    trx_id|retailer_id|         description|amount|  city_id|
+-------------------+----------+-----------+--------------------+------+---------+
|2017-11-24 19:00:00|1734117021|  644879053|unkn    ppd id: 7...|  8.58|930259917|
+-------------------+----------+-----------+--------------------+------+---------+



In [12]:
df_1 = spark.read.format("parquet").load("data/input/sales_recursive/sales_1/sales_2/2.parquet")
df_1.show()

+-------------------+----------+-----------+--------------------+------+--------+
|      transacted_at|    trx_id|retailer_id|         description|amount| city_id|
+-------------------+----------+-----------+--------------------+------+--------+
|2017-11-24 19:00:00|1734117123| 1953761884|unkn   ppd id: 15...| 19.55|45522086|
+-------------------+----------+-----------+--------------------+------+--------+



In [16]:
df_1 = spark.read.format("parquet").option("recursiveFileLookup", True).load("data/input/sales_recursive/")
df_1.show()

+-------------------+----------+-----------+--------------------+------+---------+
|      transacted_at|    trx_id|retailer_id|         description|amount|  city_id|
+-------------------+----------+-----------+--------------------+------+---------+
|2017-11-24 19:00:00|1734117123| 1953761884|unkn   ppd id: 15...| 19.55| 45522086|
|2017-11-24 19:00:00|1734117021|  644879053|unkn    ppd id: 7...|  8.58|930259917|
+-------------------+----------+-----------+--------------------+------+---------+

